In [51]:
import os
from neo4j import GraphDatabase
import pandas as pd
from ipywidgets import VBox, Button, Dropdown, Output
from IPython.display import display

# Load credentials from environment variables
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Create Neo4j driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Functions for fetching data
def get_unique_seasons():
    query = "MATCH (s:Season) RETURN s.name AS season_name ORDER BY season_name"
    with driver.session(database="neo4j") as session:
        result = session.run(query)
        return [record['season_name'] for record in result]


def get_unique_teams(season_name):
    query = """
    MATCH (squad:Squad)
    WHERE squad.name CONTAINS $season
    RETURN squad.name AS squad_name
    """
    with driver.session(database="neo4j") as session:
        result = session.run(query, parameters={'season': season_name})
        return [
            record['squad_name'].replace(season_name, "").strip()
            for record in result
        ]


def fetch_data(season_name):
    query = """
    MATCH (match:Match)-[:IS_OF_GW]->(gw:GameWeek {season: $season_name}) 
    MATCH (match)-[:HOME_TEAM]->(home_team:Squad)
    MATCH (match)-[:AWAY_TEAM]->(away_team:Squad)
    WITH 
        home_team.name AS team,
        CASE
            WHEN match.home_score > match.away_score THEN 3
            WHEN match.home_score = match.away_score THEN 1
            ELSE 0
        END AS home_points,
        away_team.name AS away_team,
        CASE
            WHEN match.away_score > match.home_score THEN 3
            WHEN match.away_score = match.home_score THEN 1
            ELSE 0
        END AS away_points
    UNWIND [
        {team: team, points: home_points},
        {team: away_team, points: away_points}
    ] AS result
    WITH result.team AS team, SUM(result.points) AS total_points
    RETURN team, total_points
    ORDER BY total_points DESC
    """
    with driver.session(database="neo4j") as session:
        result = session.run(query, parameters={"season_name": season_name})
        return pd.DataFrame([record for record in result])


# Widgets
season_dropdown = Dropdown(
    options=get_unique_seasons(),
    description="Season:",
    value=None
)

fetch_button = Button(description="Fetch Data")
output = Output()

# Event Handlers
def on_season_selected(change):
    with output:
        output.clear_output()
        print(f"Selected Season: {change['new']}")

def on_fetch_data_clicked(_):
    season_name = season_dropdown.value
    if season_name:
        with output:
            output.clear_output()
            print(f"Fetching data for {season_name}...")
            df = fetch_data(season_name)
            display(df)

# Widget Observers
season_dropdown.observe(on_season_selected, names="value")
fetch_button.on_click(on_fetch_data_clicked)

# Layout
ui = VBox([
    season_dropdown,
    fetch_button,
    output
])

# Display UI
display(ui)


ConfigurationError: URI scheme b'' is not supported. Supported URI schemes are ['bolt', 'bolt+ssc', 'bolt+s', 'neo4j', 'neo4j+ssc', 'neo4j+s']. Examples: bolt://host[:port] or neo4j://host[:port][?routing_context]